In [ ]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

prices = pd.read_csv("prices.txt", sep=r"\s+")
returns = prices.pct_change()
assets = prices.columns.tolist()

# All model fitting/selection uses only these first 250 price rows.
train_end = len(prices) - 250

pairs = [
    ("GARI", "EELT", 23.025273, -0.085821),
    ("ULXY", "FCSG", 82.495174, -1.189094),
    ("AGVF", "BENI", 27.427087, 3.450960),
    ("CTGI", "NGTE", 35.422539, 1.003893),
]

tuples = [
    ("ULXY", "FCSG", "MHRM", 104.251, -1.07821, -1.31655),
    ("GARI", "EELT", "BENI", 17.1287, -0.0784992, 0.188554),
    ("ALGO", "SPLZ", "NGTE", 52.5784, 0.160092, 0.581827),
    ("AGVF", "NAYO", "FWWG", -4.17083, 1.31506, 2.3549),
    ("RTTH", "SRTX", "CCNS", -12.6684, 0.0718073, 1.25571),
]

rolling_window = 60
z_scale = 0.75
dead_zone = 0.25

print("Train rows:", train_end)
print("Test rows:", len(prices) - train_end)
print("Mean-reversion baskets:", len(pairs) + len(tuples))


In [ ]:
def basket_components(kind, basket):
    if kind == "pair":
        y, x1, alpha, beta1 = basket
        xs = [x1]
        betas = [beta1]
        spread = prices[y] - alpha - beta1 * prices[x1]
    else:
        y, x1, x2, alpha, beta1, beta2 = basket
        xs = [x1, x2]
        betas = [beta1, beta2]
        spread = prices[y] - alpha - beta1 * prices[x1] - beta2 * prices[x2]

    return y, xs, betas, spread


def build_features_and_label(kind, basket):
    y, xs, betas, spread = basket_components(kind, basket)

    spread_mean = spread.shift(1).rolling(rolling_window).mean()
    spread_std = spread.shift(1).rolling(rolling_window).std()
    z = (spread.shift(1) - spread_mean) / spread_std.replace(0, np.nan)
    dz = z.diff()
    spread_change = spread.diff()

    features = pd.DataFrame({
        "z": z,
        "abs_z": z.abs(),
        "dz": dz,
        "z_ma5": z.rolling(5).mean(),
        "z_std20": z.rolling(20).std(),
        "spread_change_5": spread_change.shift(1).rolling(5).sum(),
    })

    gross = 1 + sum(abs(beta) for beta in betas)
    base_strength = -np.sign(z).fillna(0.0)
    base_strength = base_strength.where(z.abs() >= dead_zone, 0.0)

    base_positions = pd.DataFrame(0.0, index=prices.index, columns=assets)
    base_positions[y] = base_strength / gross
    for x, beta in zip(xs, betas):
        base_positions[x] = -base_strength * beta / gross

    base_basket_returns = (base_positions * returns).sum(axis=1)
    label = (base_basket_returns > 0).astype(int)

    return features, label, z, gross, y, xs, betas


def sharpe_ratio(return_series):
    daily_volatility = return_series.std()
    if daily_volatility <= 0 or pd.isna(daily_volatility):
        return np.nan
    return np.sqrt(250) * return_series.mean() / daily_volatility


def performance_summary(return_series):
    return pd.Series({
        "mean_daily_return": return_series.mean(),
        "daily_volatility": return_series.std(),
        "annualised_sharpe": sharpe_ratio(return_series),
        "total_return": (1 + return_series).prod() - 1,
    })


In [ ]:
basket_specs = [("pair", basket) for basket in pairs] + [("tuple", basket) for basket in tuples]

tree_positions = pd.DataFrame(0.0, index=prices.index, columns=assets)
basket_models = []

for basket_id, (kind, basket) in enumerate(basket_specs):
    features, label, z, gross, y, xs, betas = build_features_and_label(kind, basket)
    valid = features.notna().all(axis=1)
    train_mask = valid & (features.index < train_end)
    all_mask = valid

    if train_mask.sum() < 50 or label.loc[train_mask].nunique() < 2:
        continue

    tree = DecisionTreeClassifier(
        max_depth=3,
        min_samples_leaf=20,
        class_weight="balanced",
        random_state=1,
    )
    tree.fit(features.loc[train_mask], label.loc[train_mask])

    positive_probability = pd.Series(np.nan, index=prices.index)
    positive_probability.loc[all_mask] = tree.predict_proba(features.loc[all_mask])[:, 1]

    # The tree is a trade filter: only take the base mean-reversion direction
    # when the fitted probability says the trade has positive edge.
    edge = (2 * positive_probability - 1).clip(lower=0).fillna(0.0)
    base_direction = -np.sign(z).fillna(0.0)
    continuous_size = np.tanh(z.abs() / z_scale).fillna(0.0)
    strength = base_direction * continuous_size * edge
    strength = strength.where(z.abs() >= dead_zone, 0.0)

    basket_positions = pd.DataFrame(0.0, index=prices.index, columns=assets)
    basket_positions[y] = strength / gross
    for x, beta in zip(xs, betas):
        basket_positions[x] = -strength * beta / gross

    tree_positions += basket_positions / len(basket_specs)

    basket_models.append({
        "basket_id": basket_id,
        "kind": kind,
        "basket": basket,
        "train_samples": int(train_mask.sum()),
        "train_accuracy": tree.score(features.loc[train_mask], label.loc[train_mask]),
        "tree": tree,
    })

basket_model_summary = pd.DataFrame([
    {key: value for key, value in row.items() if key != "tree"}
    for row in basket_models
])

display(basket_model_summary)


In [ ]:
tree_strategy_returns = (tree_positions * returns).sum(axis=1).dropna()
train_tree_returns = tree_strategy_returns.iloc[:train_end]
test_tree_returns = tree_strategy_returns.iloc[train_end:]

summary = pd.DataFrame({
    "train": performance_summary(train_tree_returns),
    "test": performance_summary(test_tree_returns),
    "all": performance_summary(tree_strategy_returns),
})

display(summary)

cumulative_tree_returns = (1 + tree_strategy_returns).cumprod() - 1
cumulative_tree_returns.plot(title="Decision-tree-filtered mean-reversion cumulative return")


In [ ]:
# Inspect one fitted tree to see the learned filter rules.
feature_names = ["z", "abs_z", "dz", "z_ma5", "z_std20", "spread_change_5"]

if basket_models:
    first_model = basket_models[0]
    print(first_model["kind"], first_model["basket"])
    print(export_text(first_model["tree"], feature_names=feature_names))
